# GSHS El Salvador 2013 — EDA y Modelado

**Sistema de Predicción de Factores de Riesgo en Adolescentes Salvadoreños**

Pipeline del desafío:
1. Carga y limpieza (centinela SPSS → NaN)
2. EDA univariado y bivariado
3. **Tarea A:** Regresión del IMC (sin peso/altura como features)
4. **Tarea B:** Clasificación de riesgo en salud mental
5. Interpretación para políticas públicas

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Añadir src/ al path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from config import MISSING_SENTINEL, TARGET_IMC, TARGET_MENTAL_HEALTH
from data.load import load_raw_data
from data.preprocess import preprocess_data
from features.build import build_features, get_feature_columns, get_target
from models.train import (
    evaluate_classification,
    evaluate_regression,
    select_best_classification,
    select_best_regression,
)
from labels import get_label
from visualization.plots import (
    plot_age_distribution,
    plot_bmi_by_sex,
    plot_bmi_distribution,
    plot_confusion_matrix,
    plot_correlation_heatmap,
    plot_feature_importance,
    plot_mental_health_by_sex,
    plot_mental_health_prevalence,
    plot_residuals,
    plot_risk_factor_heatmap,
    plot_roc_curve,
    set_plot_style,
)
from models.train import format_feature_importances_for_report

set_plot_style()
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Carga y limpieza de datos

El valor `1.79769313486232e+308` es el máximo de float64 usado por SPSS/Stata como **dato faltante**. Se reemplaza por `np.nan` al cargar.

In [ ]:
df_raw = load_raw_data()
print(f"Registros: {len(df_raw):,} | Columnas: {df_raw.shape[1]}")
print(f"Centinela detectado en muestra Q4: {(df_raw['Q4'] >= MISSING_SENTINEL * 0.99).sum()}")

df = preprocess_data(df_raw)
print(f"\nIMC válidos: {df[TARGET_IMC].notna().sum():,}")
print(f"Riesgo mental válidos: {df[TARGET_MENTAL_HEALTH].notna().sum():,}")
print(f"Prevalencia riesgo: {df[TARGET_MENTAL_HEALTH].mean():.1%}")

df[[TARGET_IMC, TARGET_MENTAL_HEALTH, 'Q1', 'Q2']].head()

## 2. Análisis Exploratorio (EDA)

Visualizaciones de distribución de edades, IMC y prevalencia del riesgo elegido.

In [ ]:
plot_age_distribution(df, FIGURES / "eda_age_distribution.png")
plot_bmi_distribution(df, FIGURES / "eda_bmi_distribution.png")
plot_bmi_by_sex(df, FIGURES / "eda_bmi_by_sex.png")
plot_mental_health_prevalence(df, FIGURES / "eda_mental_health_prevalence.png")
plot_mental_health_by_sex(df, FIGURES / "eda_mental_health_by_sex.png")
plt.show()

# Correlaciones bivariadas: factores de riesgo vs. salud mental
risk_cols = ["QN20", "QN35", "QN38", "QN53", "QN55", "QN56", "QN57", "QN52"]
plot_risk_factor_heatmap(df, risk_cols, FIGURES / "eda_risk_factor_heatmap.png")
plt.show()

## 3. Feature Engineering

Usamos columnas **QN** (recodificaciones OMS) en lugar de **Q** para evitar colinealidad. Imputación por mediana + `RobustScaler`. Sin data leakage: excluimos Q4, Q5 y variables derivadas de peso/altura.

In [ ]:
X_reg, reg_cols, reg_preprocessor = build_features(df, "regression")
y_reg = get_target(df, "regression")

X_clf, clf_cols, clf_preprocessor = build_features(df, "classification")
y_clf = get_target(df, "classification")

print(f"Features regresión ({len(reg_cols)}): {reg_cols[:8]}...")
print(f"Features clasificación ({len(clf_cols)}): {clf_cols[:8]}...")
print(f"\nLeakage check — Q4 en regresión: {'Q4' in reg_cols}")
print(f"Leakage check — QN25 en clasificación: {'QN25' in clf_cols}")

## 4. Tarea A — Regresión IMC

Modelos: Regresión Lineal y Random Forest. Métricas: RMSE (minimizar) y R². Validación cruzada 5-fold.

In [ ]:
reg_results = evaluate_regression(X_reg, y_reg, reg_preprocessor, cv_folds=5)
best_reg = select_best_regression(reg_results)

for r in reg_results:
    print(
        f"{r.model_name:20s} | RMSE={r.rmse:.3f} | R²={r.r2:.3f} | "
        f"CV RMSE={r.cv_rmse_mean:.3f}±{r.cv_rmse_std:.3f}"
    )

print(f"\nMejor modelo: {best_reg.model_name}")
plot_residuals(best_reg.y_test, best_reg.y_pred, FIGURES / "regression_residuals.png")
plt.show()

## 5. Tarea B — Clasificación Riesgo Salud Mental

Target principal: ideación suicida seriamente considerada (**QN24**, enlace V77 del desafío). Escenario alternativo: soledad (**QN22**). Desbalance ~12% clase minoritaria. Modelos con `class_weight='balanced'` y SMOTE. Métricas: F1 minoritaria y AUC-ROC. Sin data leakage: se excluye todo el constructo QN22–QN27.

In [ ]:
clf_results = evaluate_classification(X_clf, y_clf, clf_preprocessor, cv_folds=5)
best_clf = select_best_classification(clf_results)

for r in clf_results:
    print(
        f"{r.model_name:22s} | F1(min)={r.f1_minority:.3f} | AUC={r.auc_roc:.3f} | "
        f"Acc={r.accuracy:.3f} | CV F1={r.cv_f1_mean:.3f}"
    )

print(f"\nMejor modelo: {best_clf.model_name}")
print(best_clf.classification_report)

plot_confusion_matrix(best_clf.confusion, FIGURES / "classification_confusion_matrix.png")
if best_clf.fpr is not None:
    plot_roc_curve(best_clf.fpr, best_clf.tpr, best_clf.auc_roc, FIGURES / "classification_roc_curve.png")
plt.show()

## 6. Interpretación y políticas públicas

Traducción de feature importances a acciones concretas para el MINSAL.

In [ ]:
if best_clf.feature_importances:
    translated = format_feature_importances_for_report(best_clf.feature_importances)
    plot_feature_importance(
        translated,
        FIGURES / "classification_feature_importance.png",
        title="Predictores de riesgo en salud mental (QN24)",
        use_spanish_labels=False,
    )
    plt.show()

    print("=== Recomendaciones para política pública ===\n")
    for label, imp in list(translated.items())[:5]:
        print(f"• {label}: importancia {imp:.3f}")
    print(
        "\nEl sistema propuesto permitiría al MINSAL focalizar intervenciones "
        "en escuelas con baja supervisión parental, alto consumo de alcohol "
        "y prevalencia de bullying, activando alertas tempranas antes de "
        "crisis de salud mental en adolescentes salvadoreños."
    )